In [0]:
USE workspace.gold;

WITH base_query AS (
  -- 1) Base Data with optimized joins
  SELECT
    f.order_number,
    f.order_date,
    f.sales_amount,
    f.quantity,
    f.customer_key,
    c.country,
    p.category,
    p.subcategory,
    p.product_name
  FROM
    gold.fact_sales f
      INNER JOIN gold.dim_customers c
        ON f.customer_key = c.customer_key
      INNER JOIN gold.dim_products p
        ON f.product_key = p.product_key
  WHERE
    f.order_date IS NOT NULL
),
aggregated AS (
  -- 2) Core Aggregation with pre-computed date
  SELECT
    DATE_TRUNC('month', order_date) AS order_month_date,
    DATE_FORMAT(order_date, 'MMM-yyyy') AS order_month,
    country,
    category,
    subcategory,
    product_name,
    COUNT(DISTINCT order_number) AS total_orders,
    COUNT(DISTINCT customer_key) AS total_customers,
    SUM(sales_amount) AS total_sales,
    SUM(quantity) AS total_quantity
  FROM
    base_query
  GROUP BY
    ALL
),
category_metrics AS (
  -- 3) Pre-compute category-level metrics once
  SELECT
    category,
    SUM(total_customers) AS category_total_customers,
    SUM(total_orders) AS category_total_orders
  FROM
    aggregated
  GROUP BY
    category
),
overall_metrics AS (
  -- 4) Pre-compute overall metrics once
  SELECT
    SUM(total_customers) AS overall_customers
  FROM
    aggregated
),
final AS (
  -- 5) Join pre-computed metrics efficiently
  SELECT
    a.*,
    CASE
      WHEN a.total_orders = 0 THEN 0
      ELSE a.total_sales / a.total_orders
    END AS avg_order_value,
    cm.category_total_customers,
    om.overall_customers,
    ROUND(
      (CAST(cm.category_total_customers AS DOUBLE) / om.overall_customers) * 100,
      2
    ) AS pct_customers_by_category,
    RANK() OVER (PARTITION BY a.order_month_date ORDER BY a.total_sales DESC) AS country_rank
  FROM
    aggregated a
      CROSS JOIN overall_metrics om
      LEFT JOIN category_metrics cm
        ON a.category = cm.category
)
SELECT
  *
FROM
  final
ORDER BY
  order_month_date